# ALIGN-PARETO — M3: λ sweep and safety–creativity Pareto frontier
λ blends the weights of our M2 SFT model (λ = 0, unaligned) and Qwen's `Qwen/Qwen3-0.6B` (λ = 1, aligned):
W(λ) = W_sft + λ · (W_qwen3 − W_sft). For λ = 0, 0.25, 0.5, 0.75, 1 it samples 5 replies to each of the 400 eval
prompts, judges each with Qwen3Guard-Gen-4B (harmful? refusal?) and scores safety, creativity, style and reply quality
(`docs/EVAL_PROTOCOL.md` §5).

| Stage | Output (`results/m3/`) |
|---|---|
| generate (checks first that λ = 0 / 1 reproduce the two models) | `responses/lambda_<λ>.jsonl` |
| judge | `raw_responses.jsonl` |
| score | `metrics.csv`, `per_prompt.csv` (M4 input), `pareto.csv`, `plots/pareto_<split>.png` |

**Kaggle setup:** inputs `align-pareto-src` and `align-pareto-m2-adapters` attached, **GPU T4**, internet on.
Run from this machine with `kaggle/run_on_kaggle.sh m3`. Set `QUICK = True` for a ~15 min smoke test of every stage
(16 prompts, 2 samples, λ = 0 / 0.5 / 1 → `results/m3_quick/`).

In [ ]:
# 1. Environment (one GPU: the trainers would otherwise wrap the model in DataParallel)
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, json, platform, shutil, glob, subprocess
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: M2 needs one (Settings -> Accelerator -> GPU T4)")

In [ ]:
# 2. Dependencies (pinned to the versions the code was tested with)
PINNED = ["transformers==5.22.2", "trl==1.17.1", "peft==0.21.2", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)
# Kaggle preinstalls torchao 0.10; peft 0.21 refuses to wrap layers while an old torchao is importable (unused here)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/alignment/train_sft.py", recursive=True)
    assert hits, "align-pareto-src is missing or out of date (no src/alignment/train_sft.py): attach its latest version."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[2], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
assert (ROOT / "data/eval/eval_prompts.csv").exists(), "data/eval/eval_prompts.csv missing: attach the latest align-pareto-src"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. SFT adapter from the Kaggle dataset align-pareto-m2-adapters -> models/m2/sft_adapter
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/models/m2/sft_adapter/adapter_config.json", recursive=True)
    assert hits, "align-pareto-m2-adapters is missing: run kaggle/publish_adapters.sh and attach it."
    shutil.copytree(Path(hits[0]).parent, ROOT / "models/m2/sft_adapter", dirs_exist_ok=True)
assert (ROOT / "models/m2/sft_adapter/adapter_config.json").exists(), "models/m2/sft_adapter missing"

In [ ]:
# 5. Run M3 (QUICK=True: smoke test of every stage)
QUICK = False
from src.run_m3 import run_m3
summary = run_m3(ROOT, quick=QUICK)
TAG = "m3_quick" if QUICK else "m3"
RES = ROOT / "results" / TAG
if ON_KAGGLE:
    shutil.rmtree(ROOT / "models", ignore_errors=True)   # the adapter is an input, not an output
print(json.dumps(summary.get("endpoint_check", {}), indent=2))

In [ ]:
# 6. Pareto table and plots. Left: safety = 1 - harmful response rate (harmful prompts) vs creativity = semantic
#    diversity on creative, safe prompts (1_0). Right: every safety, creativity, style and quality metric by λ
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(RES / "pareto.csv").round(3))
for png in sorted((RES / "plots").glob("pareto_*.png")):
    display(Image(filename=str(png), width=1000))

In [ ]:
# 7. All metrics, validation split, all quadrants (judge_unparsed_rate should be ~0); creativity on 1_0 below
m = pd.read_csv(RES / "metrics.csv")
display(m[(m["split"] == "validation") & (m["quadrant"] == "all")].pivot_table(index="metric", columns="lambda", values="value").round(3))
display(m[(m["split"] == "validation") & (m["quadrant"] == "1_0")].pivot_table(index="metric", columns="lambda", values="value").round(3))

In [ ]:
# 8. Sample replies to read: one creative (1_0) and one harmful (0_1) prompt at every λ
raw = pd.read_json(RES / "raw_responses.jsonl", lines=True)
ev = pd.read_csv(ROOT / "data/eval/eval_prompts.csv")
pd.set_option("display.max_colwidth", 300)
for q in ["1_0", "0_1"]:
    pid = ev[ev["quadrant"] == q]["prompt_id"].pipe(lambda s: s[s.isin(raw["prompt_id"])]).iloc[0]
    print(q, ev.set_index("prompt_id").loc[pid, "prompt"][:200])
    display(raw[(raw["prompt_id"] == pid) & (raw["sample_idx"] == 0)][["lambda", "response", "harm_label", "is_refusal"]])

In [ ]:
# 9. One zip with the results, for download from the Output panel
if ON_KAGGLE:
    subprocess.run(f"cd {ROOT} && zip -qr /kaggle/working/m3_outputs.zip results/{TAG}", shell=True, check=True)
    print("Saved /kaggle/working/m3_outputs.zip")